# Task 2 — Evasion-Based Clarity Classification

Fine-tuning **Llama 3.2 3B-Instruct** with 4-bit QLoRA + DoRA to classify the political evasion strategy into nine fine-grained categories, then map the prediction back to the three clarity macro-categories.


## Environment setup

Import dependencies, configure paths, and detect whether the notebook is running locally or in Colab.


In [ ]:
# Install dependencies if running on Colab
# !pip install -q -U transformers peft trl bitsandbytes datasets accelerate

import os, sys, json, gc
import torch
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from huggingface_hub import login
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig
from datasets import load_from_disk
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix

try:
    import yaml
except ImportError as exc:
    raise ImportError("Missing dependency: PyYAML. Install with `pip install pyyaml`.") from exc

# Path configuration and environment detection (Colab / local)
try:
    import google.colab
    from google.colab import drive, userdata
    drive.mount('/content/drive', force_remount=True)

    # Base paths on Google Drive
    BASE_DIR = "/content/drive/MyDrive/progettoLLM"
    REPO_DIR = os.path.join(BASE_DIR, "CLARITY")
    hf_cache_dir = os.path.join(BASE_DIR, "hf_cache")
    
    os.makedirs(hf_cache_dir, exist_ok=True)
    os.environ["HF_HOME"] = hf_cache_dir
    
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    # Login HF tramite Colab Secrets
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    login(token=hf_token)
    print(f"Ambiente Google Colab configurato. Cache: {hf_cache_dir}")

except ImportError:
    # Percorsi base in locale
    BASE_DIR = ".."
    REPO_DIR = BASE_DIR
    
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    # Lettura del token da file .env locale
    env_path = os.path.join(REPO_DIR, ".env")
    if not os.path.exists(env_path):
        env_path = ".env"
        
    if os.path.exists(env_path):
        with open(env_path) as f:
            for line in f:
                if line.startswith("HF_TOKEN="):
                    hf_token = line.split("=", 1)[1].strip().strip("'\"")
                    os.environ['HF_TOKEN'] = hf_token
                    login(token=hf_token)
                    break
    print("Ambiente locale rilevato.")

print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


## Configuration, paths, and prompt

Load the project configuration, define model paths, and set the Task 2 instruction prompt.


In [ ]:
# Config from file
CONFIG_PATH = os.path.join(REPO_DIR, "config", "config.yml")
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f) or {}

# Control flag
ESEGUI_TRAINING = bool(config.get("training", {}).get("enabled", True))

# Model and paths
MODEL_ID = config.get("model", {}).get("id", "meta-llama/Llama-3.2-3B-Instruct")

# Base results directory for Task 2
RES_DIR = os.path.join(REPO_DIR, config.get("paths", {}).get("results_dir", "results"))

TASK2_DIR = os.path.join(REPO_DIR, config.get("paths", {}).get("task2_dir", "results/task2"))

# Dynamically built task-specific paths
OUTPUT_DIR = os.path.join(TASK2_DIR, "checkpoints")
PATH_MODELLO_SALVATO = os.path.join(TASK2_DIR, "modello_finale")
RISULTATI_DIR = os.path.join(TASK2_DIR, "valutazione")

# Training params (Task 2)
TRAINING_CONFIG = config.get("training", {}).get("task2", {})
SEED = config.get("training", {}).get("seed", 42)
PER_DEVICE_TRAIN_BATCH_SIZE = TRAINING_CONFIG.get("batch_size", 4)
GRADIENT_ACCUMULATION_STEPS = TRAINING_CONFIG.get("gradient_accumulation_steps", 2)
NUM_TRAIN_EPOCHS = TRAINING_CONFIG.get("num_train_epochs", 1)
MAX_LENGTH = TRAINING_CONFIG.get("max_length", 512)

# Dataset/augmentation config
DATASET_CONFIG = config.get("dataset", {})
DATASET_USE_AUGMENTED = bool(DATASET_CONFIG.get("use_augmented", False))
AUGMENTATION_NAME = DATASET_CONFIG.get("augmentation", {}).get("name", "none")
AUGMENTATION_PARAMS = DATASET_CONFIG.get("augmentation", {}).get("params", {})

# Precision: bfloat16 when supported, otherwise float16
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"Compute dtype: {COMPUTE_DTYPE}")

LABELS = ["Ambivalent", "Clear Reply", "Clear Non-Reply"]

# System prompt for the nine Task 2 categories
SYSTEM_PROMPT_TASK2 = """You are an expert political communication analyst. Classify the politician's answer to the question by choosing EXACTLY ONE of the following 9 categories:

1. 'Explicit'
2. 'Declining to answer'
3. 'Claims ignorance'
4. 'Dodging'
5. 'Deflection'
6. 'General'
7. 'Implicit'
8. 'Partial/half-answer'
9. 'Clarification'

Classification examples (few-shot):
Q: "Will you support the new tax law?"
A: "I am not aware of the details of the law." -> {"category": "Claims ignorance"}

Q: "What do you think about your party's scandal?"
A: "You should look at what the opposition did last year!" -> {"category": "Deflection"}

Q: "What measures will you take on climate?"
A: "Climate is important, and we will do our best for everyone." -> {"category": "General"}

Reply ONLY with a valid JSON object containing the single key "category". Do not add markdown formatting, preambles, or explanations.
"""
print("Configuration loaded.")


## Dataset preparation

Load the QEvasion splits, optionally apply augmentation, and format examples with the Llama chat template.


In [ ]:
from datasets import DatasetDict
from src.data.augmentation import get_augmentation_fn
from src.data.label_utils import build_label_maps

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load train and test splits using absolute paths
TRAIN_DATA_PATH = os.path.join(REPO_DIR, DATASET_CONFIG.get("train_path", "dataset/QEvasion/train"))
TEST_DATA_PATH = os.path.join(REPO_DIR, DATASET_CONFIG.get("test_path", "dataset/QEvasion/test"))

train_dataset = load_from_disk(TRAIN_DATA_PATH)
test_dataset  = load_from_disk(TEST_DATA_PATH)
print(f"Train: {len(train_dataset)} examples | Test: {len(test_dataset)} examples")

if DATASET_USE_AUGMENTED:
    if "evasion_label" not in train_dataset.column_names:
        print("Augmentation skipped: 'evasion_label' not found in train split.")
    else:
        ds = DatasetDict({"train": train_dataset, "validation": test_dataset, "test": test_dataset})
        label2id, _ = build_label_maps(ds)
        augment_fn = get_augmentation_fn(AUGMENTATION_NAME)
        ds = augment_fn(ds, label2id, **AUGMENTATION_PARAMS)
        train_dataset = ds["train"]
        print(f"Augmentation applied: {AUGMENTATION_NAME}")

def format_prompt_task2(example):
    """Format one example with the Llama 3.1 chat template for Task 2."""
    question = example.get('interview_question', '')
    answer = example.get('interview_answer', '')
    label = example.get('evasion_label', 'Explicit')

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_TASK2},
        {"role": "user", "content": f"Question: {question}\nPolitician answer: {answer}"},
        {"role": "assistant", "content": str(label)},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

# Applica la formattazione solo al train set (il test usa la funzione di inferenza)
formatted_train = train_dataset.map(format_prompt_task2, remove_columns=train_dataset.column_names)

print("\n--- Formatted prompt example ---")
print(formatted_train[0]["text"][:500] + "\n...")


## Model training or loading

Either fine-tune the model with QLoRA/DoRA adapters or load the saved Task 2 adapter.


In [ ]:
# Clear VRAM before loading the model
torch.cuda.empty_cache()
gc.collect()

# Heuristic: leave some headroom on GPU and allow CPU offload if needed
if torch.cuda.is_available():
    total_gb = int(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3))
    gpu_gb = max(total_gb - 2, 1)
    MAX_MEMORY = {0: f"{gpu_gb}GiB", "cpu": "64GiB"}
else:
    MAX_MEMORY = {"cpu": "64GiB"}

# Shared 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    llm_int8_enable_fp32_cpu_offload=True,
)

if ESEGUI_TRAINING:
    # ── TRAINING ──────────────────────────────────────────────
    print("Training mode enabled.")

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="auto",
        max_memory=MAX_MEMORY,
        quantization_config=bnb_config,
        torch_dtype=COMPUTE_DTYPE,
    )
    model = prepare_model_for_kbit_training(model)
    
    # Enable gradient checkpointing to reduce VRAM usage
    model.gradient_checkpointing_enable()

    # DoRA configuration for attention layers
    peft_config = LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM", use_dora=True,
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    training_args = SFTConfig(           
        output_dir=OUTPUT_DIR,
        
        # Per-device batch size
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        
        # Gradient accumulation defines the effective batch size
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        
        # Optimizer
        optim="adamw_8bit",                # Usa questo (o "adamw_torch"). "paged_" è lento.
        
        # Data loading parallelism
        dataloader_num_workers=2,          
        
        save_strategy="epoch",
        logging_steps=10,
        learning_rate=2e-4,
        weight_decay=0.001,
        fp16=(COMPUTE_DTYPE == torch.float16),
        bf16=(COMPUTE_DTYPE == torch.bfloat16),
        max_grad_norm=0.3,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        warmup_steps=100,
        group_by_length=True,
        lr_scheduler_type="cosine",
        report_to="none",
        max_length=MAX_LENGTH,
        seed=SEED,
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=formatted_train,
        args=training_args,
        peft_config=None,          
        processing_class=tokenizer,
        # max_seq_length is handled by SFTConfig
    )

    print("Starting training...")
    trainer.train()

    # Save adapters and tokenizer
    trainer.model.save_pretrained(PATH_MODELLO_SALVATO)
    tokenizer.save_pretrained(PATH_MODELLO_SALVATO)
    print(f"Model saved to: {PATH_MODELLO_SALVATO}")

else:
    # ── CARICAMENTO DA CHECKPOINT ─────────────────────────────
    print(f"Loading base model ({MODEL_ID}) in 4-bit...")

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="auto",
        max_memory=MAX_MEMORY,
        quantization_config=bnb_config,
        torch_dtype=COMPUTE_DTYPE,
    )

    print(f"Applying LoRA adapters from: {PATH_MODELLO_SALVATO}")
    model = PeftModel.from_pretrained(
        base_model, PATH_MODELLO_SALVATO,
        low_cpu_mem_usage=True, device_map="auto",
    )

    # Sovrascriviamo il tokenizer con quello salvato col checkpoint
    tokenizer = AutoTokenizer.from_pretrained(PATH_MODELLO_SALVATO)
    print("Model and adapters loaded.")

model.eval()
print(f"Model in eval mode. Device: {model.device}")


## Inference helpers and label mapping

Predict the nine-way evasion label and map it to the three clarity classes.


In [ ]:
import json
import torch
from collections import Counter

MAPPING_9_CLASSES = {
    "Explicit": "Clear Reply",
    "Declining to answer": "Clear Non-Reply",
    "Claims ignorance": "Clear Non-Reply",
    "Clarification": "Ambivalent",
    "Dodging": "Ambivalent",
    "Deflection": "Ambivalent",
    "General": "Ambivalent",
    "Implicit": "Ambivalent",
    "Partial/half-answer": "Ambivalent"
}
LABELS_9_CLASSES = list(MAPPING_9_CLASSES.keys())
ANNOTATOR_COLUMNS = ["annotator1", "annotator2", "annotator3"]

def get_true_evasion_label(example):
    """Recupera la label Task 2 reale o la ricostruisce dai voti degli annotatori."""
    label = str(example.get('evasion_label', '')).strip()
    if label in LABELS_9_CLASSES:
        return label

    votes = [
        str(example.get(col, '')).strip()
        for col in ANNOTATOR_COLUMNS
        if str(example.get(col, '')).strip() in LABELS_9_CLASSES
    ]
    if not votes:
        return ""

    vote_counts = Counter(votes)
    max_votes = max(vote_counts.values())
    candidates = [label for label, count in vote_counts.items() if count == max_votes]
    if len(candidates) == 1:
        return candidates[0]

    clarity = str(example.get('clarity_label', '')).strip()
    coherent = [label for label in candidates if MAPPING_9_CLASSES.get(label) == clarity]
    return coherent[0] if coherent else candidates[0]

def predict_evasion(example):
    """
    Generate the prediction by extracting the category from the generated text
    and automatically map it to the clarity macro-category.
    """
    question = example.get('interview_question', '')
    answer = example.get('interview_answer', '')

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_TASK2},
        {"role": "user", "content": f"Question: {question}\nAnswer: {answer}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode generated text
    generated_ids = outputs[0][inputs['input_ids'].shape[-1]:]
    raw_output = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    
    # Default fallback
    predicted_technique = "Dodging"
    
    # Category extraction
    # Check whether any of the nine labels appears in the generated text
    for category in MAPPING_9_CLASSES.keys():
        if category.lower() in raw_output.lower():
            predicted_technique = category
            break
            
    # Map to the macro class
    macro_clarity = MAPPING_9_CLASSES.get(predicted_technique, "Ambivalent")
    
    return predicted_technique, macro_clarity, raw_output

# ============================================================
# Quick sanity check
# ============================================================
print("--- Sanity check (3 esempi) ---")
for i in range(min(3, len(test_dataset))):
    es = test_dataset[i]
    technique, clarity, raw = predict_evasion(es)
    true_label = str(es.get('clarity_label', '')).strip()
    print(f"[{i+1}] Raw: {raw:40s} \n    -> Technique: {technique:22s} -> Clarity: {clarity:15s} (True: {true_label})\n")


## Evaluation and outputs

Evaluate the mapped clarity predictions, plot the confusion matrix, and save test-set predictions.


In [ ]:
y_true_clarity = []
y_pred_clarity = []
y_true_raw_evasion = []
y_pred_raw_evasion = []

print(f"Evaluating {len(test_dataset)} examples...")

for i in tqdm(range(len(test_dataset))):
    example = test_dataset[i]

    # True macro-category label
    y_true_clarity.append(str(example.get('clarity_label', '')).strip())
    y_true_raw_evasion.append(get_true_evasion_label(example))

    # Prediction, normalization, and mapping
    technique, clarity, raw = predict_evasion(example)
    y_pred_raw_evasion.append(technique)
    y_pred_clarity.append(clarity)

# Pre-mapping classification report over the nine Task 2 classes
print("\n" + "=" * 60)
print("PRE-MAPPING REPORT — TASK 2 (9 EVASION CLASSES)")
print("=" * 60)
if any(label in LABELS_9_CLASSES for label in y_true_raw_evasion):
    print(classification_report(
        y_true_raw_evasion,
        y_pred_raw_evasion,
        labels=LABELS_9_CLASSES,
        digits=3,
        zero_division=0,
    ))
else:
    print("True Task 2 labels are not available in the test set: skipping the 9-class report.")
    print("Pre-mapping prediction distribution:")
    print(pd.Series(y_pred_raw_evasion).value_counts())

# Classification Report (3 cifre decimali)
print("\n" + "=" * 60)
print("FINAL REPORT — TASK 2 (EVASION -> CLARITY)")
print("=" * 60)
print(classification_report(y_true_clarity, y_pred_clarity, labels=LABELS, digits=3))

# Confusion matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_true_clarity, y_pred_clarity, labels=LABELS)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABELS, yticklabels=LABELS)
plt.xlabel('Predictions (via Task 2 mapping)')
plt.ylabel('True labels')
plt.title('Confusion Matrix — Evasion-Based Strategy (Task 2)')
plt.tight_layout()

# Save plot
os.makedirs(RISULTATI_DIR, exist_ok=True)
percorso_grafico = os.path.join(RISULTATI_DIR, "matrice_confusione.png")
plt.savefig(percorso_grafico, bbox_inches='tight', dpi=300)
plt.show()

# Predicted evasion-technique distribution
df_raw = pd.Series(y_pred_raw_evasion).value_counts().reset_index()
df_raw.columns = ['Task 2 Technique', 'Count']

print("\n--- Raw prediction distribution (Task 2) ---")
print(df_raw)

# Save predictions to CSV
print(f"\nSaving results to: {RISULTATI_DIR}...")
df_risultati = pd.DataFrame({
    'Domanda': [ex.get('interview_question', '') for ex in test_dataset],
    'Risposta_Politico': [ex.get('interview_answer', '') for ex in test_dataset],
    'Vero_Task1_Clarity': y_true_clarity,
    'Vero_Task2_Evasion': y_true_raw_evasion,
    'Predetto_Task2_Evasion': y_pred_raw_evasion,
    'Predetto_Task1_Clarity': y_pred_clarity
})

csv_output_path = os.path.join(RISULTATI_DIR, "predizioni_test_set.csv")
df_risultati.to_csv(csv_output_path, index=False)
print(f"CSV saved successfully: {csv_output_path}")


## Label diagnostics

Inspect raw evasion labels and saved prediction distributions.


In [ ]:
import os
import pandas as pd
from collections import Counter

print("\n=== Task 2 diagnostics ===")
print("Test set evasion_label (from dataset):")
print(dict(Counter(test_dataset["evasion_label"])))

csv_path = os.path.join(RISULTATI_DIR, "predizioni_test_set.csv")
if not os.path.exists(csv_path):
    print(f"CSV not found: {csv_path}")
else:
    df = pd.read_csv(csv_path)
    print("\nNull values in Vero_Task2_Evasion:", df["Vero_Task2_Evasion"].isna().sum())
    print("Vero_Task2_Evasion (value_counts):")
    print(df["Vero_Task2_Evasion"].fillna("<NA>").value_counts())
    print("Predetto_Task2_Evasion (value_counts):")
    print(df["Predetto_Task2_Evasion"].value_counts())

    if {"Vero_Task1_Clarity", "Predetto_Task1_Clarity"}.issubset(df.columns):
        y_true = df["Vero_Task1_Clarity"].astype(str).str.strip()
        y_pred = df["Predetto_Task1_Clarity"].astype(str).str.strip()
        print("\nMacro report (from CSV):")
        print(classification_report(y_true, y_pred, labels=LABELS, digits=3))
